### 1. Data Ingestion: Steam Games CSV
Reading the static historical dataset. Applied `escape` and `multiLine` options to handle complex game descriptions and prevent column shifting.

In [0]:
file_path = "/Volumes/workspace/default/raw_data/games.csv"

steam_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("escape", "\"") \
    .option("multiLine", "true") \
    .load(file_path)

In [0]:
steam_df.printSchema()
print(f"Total rows ingested: {steam_df.count()}")
print(f"Total columns: {len(steam_df.columns)}")

In [0]:
display(steam_df.limit(10))

In [0]:
from pyspark.sql.functions import col, sum as _sum

display(
    steam_df.select([
        _sum(col(c).isNull().cast("int")).alias(c) for c in steam_df.columns
    ])
)

### 2. External API Integration
Fetching live current player counts for specific popular titles from the external Steam API to enrich our dataset.

In [0]:
import requests
from pyspark.sql.types import StructType, StructField, IntegerType

# We pick a few known AppIDs (e.g., CS2, Dota 2, PUBG) to demonstrate API integration
target_app_ids = [730, 570, 578080, 440, 322330]
api_records = []

for app_id in target_app_ids:
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={app_id}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            player_count = data.get("response", {}).get("player_count", 0)
            api_records.append((app_id, player_count))
    except Exception as e:
        print(f"Failed to fetch data for {app_id}: {e}")

# Create DataFrame from API results
api_schema = StructType([
    StructField("AppID", IntegerType(), True),
    StructField("CurrentPlayers", IntegerType(), True)
])

api_df = spark.createDataFrame(api_records, api_schema)
display(api_df)

In [0]:
from pyspark.sql.functions import col

games_names = steam_df.select("AppID", "Name").where(col("AppID").isin(target_app_ids))
display(games_names)

### 3. Data Transformations
Practicing basic Spark operations: `filter` to clean data, `select` for relevant columns, `join` with API data, and `groupBy` for analytical aggregations.

In [0]:
from pyspark.sql.functions import round, avg, count, split, explode, trim, when

# Filter invalid records, unnest genres into individual rows, and cast types
cleaned_df = steam_df \
    .filter(col("Name").isNotNull()) \
    .filter(~col("Name").contains("Playtest")) \
    .filter(col("Genres").isNotNull() & (col("Genres") != "[null]")) \
    .select(
        "AppID", 
        "Name", 
        col("Price").cast("double").alias("Price"), 
        col("Peak CCU").cast("long").alias("PeakCCU"),
        explode(split(col("Genres"), ",")).alias("Genre")
    ) \
    .withColumn("Genre", trim(col("Genre")))

enriched_df = cleaned_df.join(api_df, on="AppID", how="left")

# Calculate market metrics per genre
genre_stats_df = enriched_df \
    .groupBy("Genre") \
    .agg(
        count("*").alias("TotalGames"),
        round(avg("Price"), 2).alias("AveragePrice_USD"),
        round(
            avg(when(col("PeakCCU") > 0, col("PeakCCU"))), 
            1
        ).alias("AvgPeakPlayers")
    ) \
    .filter(col("TotalGames") >= 100) \
    .orderBy(col("TotalGames").desc())

In [0]:
print("API Enrichment verification:")
display(
    enriched_df
    .filter(col("CurrentPlayers").isNotNull())
    .select("AppID", "Name", "CurrentPlayers")
    .distinct()
    .sort(col("CurrentPlayers").desc())
)

In [0]:
print("Aggregated Genre Metrics:")
display(genre_stats_df.sort(col("AvgPeakPlayers").desc()).limit(10))

### 4. Save to Delta Table

In [0]:


%sql CREATE SCHEMA IF NOT EXISTS dbr_dev_ua5816bd.dan;

In [0]:
user_login = "dan"
table_name = f"dbr_dev_ua5816bd.{user_login}.steam_genre_stats"

genre_stats_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print(f"Successfully saved to {table_name}")
